# target speaker extraction demo

**use the `.venv` kernel from `uv sync`**

shows our extraction pipeline on a hard example: 2 speakers, 50% overlap, equal loudness (SIR=0dB). Whisper alone gets 91% WER, with extraction it drops to 9%.

In [ ]:
import os, sys
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import torch
import jiwer
import soundfile as sf
import IPython.display as ipd
from transformers import WhisperProcessor, WhisperForConditionalGeneration

from src.preprocess import preprocess_audio, normalize_text, TARGET_SAMPLE_RATE
from src.extraction_net import ExtractionNet, compute_stft, apply_mask_and_istft
from src.speaker_encoder import SpeakerEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## load models

In [ ]:
# Load extraction network
model = ExtractionNet(input_power=0.3).to(device)
ckpt = torch.load("checkpoints/hybrid_alpha_0.1/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Extraction model loaded")

# Load speaker encoder
speaker_encoder = SpeakerEncoder(device=device)
print("Speaker encoder loaded")

# Load Whisper
whisper_proc = WhisperProcessor.from_pretrained("openai/whisper-tiny")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
whisper_model.eval()
print("Whisper loaded")

In [ ]:
def transcribe(audio_array):
    """Transcribe a numpy audio array using Whisper."""
    inputs = whisper_proc(audio_array, sampling_rate=TARGET_SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    forced_decoder_ids = whisper_proc.get_decoder_prompt_ids(language="english", task="transcribe")
    with torch.no_grad():
        predicted_ids = whisper_model.generate(inputs["input_features"], forced_decoder_ids=forced_decoder_ids)
    return normalize_text(whisper_proc.batch_decode(predicted_ids, skip_special_tokens=True)[0])

## demo sample

mix_librispeech_04146 -- 2 speakers at equal loudness, 50% overlap
- target: "the king who had from this moment become in reality the principal dancer in the quadrille cast a look upon his vanquished rival"
- interferer: "ojo examined this curious contrivance with wonder"

In [ ]:
# File paths
MIX_DIR = "data/synthetic_mixtures/librispeech_extraction"
mix_id = "mix_librispeech_04146"

mix_path = os.path.join(MIX_DIR, f"{mix_id}.wav")
target_path = "data/librispeech/LibriSpeech/test-clean/7127/75946/7127-75946-0029.flac"
enrollment_path = "data/librispeech/LibriSpeech/test-clean/7127/75947/7127-75947-0019.flac"
noise_path = "data/librispeech/LibriSpeech/test-clean/1284/1181/1284-1181-0000.flac"

reference_text = normalize_text("the king who had from this moment become in reality the principal dancer in the quadrille cast a look upon his vanquished rival")
print(f"Reference: {reference_text}")

### step 1: mixture audio
two speakers talking over each other

In [ ]:
mix_audio, _ = sf.read(mix_path, dtype="float32")
print("Mixture audio (2 speakers, SIR=0 dB, 50% overlap):")
ipd.Audio(mix_audio, rate=TARGET_SAMPLE_RATE)

### step 2: clean target
what the target speaker sounds like alone

In [ ]:
target_audio = preprocess_audio(target_path).squeeze(0).numpy()
print("Clean target audio:")
ipd.Audio(target_audio, rate=TARGET_SAMPLE_RATE)

### step 3: whisper on raw mixture (baseline)

In [ ]:
baseline_pred = transcribe(mix_audio)
baseline_wer = jiwer.wer(reference_text, baseline_pred)

print(f"Predicted:  {baseline_pred}")
print(f"Reference:  {reference_text}")
print(f"\nBaseline WER: {baseline_wer:.0%}")

### step 4: enrollment clip
different utterance, same speaker. ECAPA-TDNN extracts a 192-dim embedding from this to tell the model who to isolate.

In [ ]:
enrollment_audio = preprocess_audio(enrollment_path).squeeze(0).numpy()
print("Enrollment clip (same speaker, different utterance):")
ipd.Audio(enrollment_audio, rate=TARGET_SAMPLE_RATE)

In [ ]:
# Extract speaker embedding
speaker_emb = speaker_encoder.encode_file(enrollment_path)
print(f"Speaker embedding shape: {speaker_emb.shape}")

### step 5: run extraction
predict a time-frequency mask conditioned on the speaker embedding, apply it to the mixture

In [ ]:
# STFT of mixture
waveform = preprocess_audio(mix_path)
magnitude, phase = compute_stft(waveform)

# Predict mask
with torch.no_grad():
    emb = speaker_emb.unsqueeze(0).to(device)
    mask = model(magnitude.to(device), emb)

# Apply mask and reconstruct
extracted = apply_mask_and_istft(magnitude.to(device), phase.to(device), mask)
extracted_audio = extracted.squeeze(0).cpu().numpy()

print("Extracted audio (target speaker isolated):")
ipd.Audio(extracted_audio, rate=TARGET_SAMPLE_RATE)

### step 6: whisper on extracted audio

In [ ]:
extracted_pred = transcribe(extracted_audio)
extracted_wer = jiwer.wer(reference_text, extracted_pred)

print(f"Predicted:  {extracted_pred}")
print(f"Reference:  {reference_text}")
print(f"\nExtracted WER: {extracted_wer:.0%}")

### summary

In [ ]:
print(f"{'Condition':<25} {'WER':>8}  Transcription")
print("-" * 90)
print(f"{'Baseline (raw mixture)':<25} {baseline_wer:>7.0%}  {baseline_pred}")
print(f"{'With extraction':<25} {extracted_wer:>7.0%}  {extracted_pred}")
print(f"{'Reference':<25} {'':>8}  {reference_text}")
print(f"\nWER reduction: {baseline_wer:.0%} -> {extracted_wer:.0%}")